<a href="https://colab.research.google.com/github/melike0019/GTZAN-Audio-Classification/blob/melike/02_Dataset_Loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#PyTorch Dataset ve DataLoader

**Amaç:** Partner'ın `data.ipynb` defterinde işlenip kaydedilen mel-spectrogram .npy dosyalarını PyTorch Dataset sınıfına sarmalamak, augmentation eklemek ve DataLoader ile batch akışını kurmak.

**Bağımlılıklar:**
- `Processed_Data/X_train.npy, y_train.npy` (z-score normalize edilmiş)
- `Processed_Data/X_val.npy, y_val.npy`
- `Processed_Data/X_test.npy, y_test.npy`
- `Processed_Data/mean_std.npy`


In [2]:
import numpy as np
import os

PROC = "/content/drive/MyDrive/gtzan_project/Processed_Data"

# Hepsini yükle
X_train = np.load(f"{PROC}/X_train.npy")
y_train = np.load(f"{PROC}/y_train.npy")
X_val   = np.load(f"{PROC}/X_val.npy")
y_val   = np.load(f"{PROC}/y_val.npy")
X_test  = np.load(f"{PROC}/X_test.npy")
y_test  = np.load(f"{PROC}/y_test.npy")
mean_std = np.load(f"{PROC}/mean_std.npy")

print("=== Veri seti istatistikleri ===")
print(f"Train: {X_train.shape}, etiketler: {y_train.shape}")
print(f"Val:   {X_val.shape}, etiketler: {y_val.shape}")
print(f"Test:  {X_test.shape}, etiketler: {y_test.shape}")
print(f"\nVeri tipi: {X_train.dtype}")
print(f"Değer aralığı (train): [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"Mean: {X_train.mean():.4f}, Std: {X_train.std():.4f}")
print(f"\n=== Sınıf dağılımı ===")
print(f"Train: {np.bincount(y_train)}")
print(f"Val:   {np.bincount(y_val)}")
print(f"Test:  {np.bincount(y_test)}")
print(f"\nNormalizasyon parametreleri (mean_std.npy): {mean_std}")

=== Veri seti istatistikleri ===
Train: (699, 128, 1292), etiketler: (699,)
Val:   (150, 128, 1292), etiketler: (150,)
Test:  (150, 128, 1292), etiketler: (150,)

Veri tipi: float32
Değer aralığı (train): [-2.394, 2.754]
Mean: -0.0000, Std: 1.0000

=== Sınıf dağılımı ===
Train: [70 70 70 70 70 69 70 70 70 70]
Val:   [15 15 15 15 15 15 15 15 15 15]
Test:  [15 15 15 15 15 15 15 15 15 15]

Normalizasyon parametreleri (mean_std.npy): [-42.798103  15.542746]


In [3]:
GENRES = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]

print("Train sınıf sayıları:")
for i, g in enumerate(GENRES):
    count = np.sum(y_train == i)
    marker = " ← eksik" if count < 70 else ""
    print(f"  {i}: {g:12s} → {count}{marker}")

Train sınıf sayıları:
  0: blues        → 70
  1: classical    → 70
  2: country      → 70
  3: disco        → 70
  4: hiphop       → 70
  5: jazz         → 69 ← eksik
  6: metal        → 70
  7: pop          → 70
  8: reggae       → 70
  9: rock         → 70


**Veri kontrolü tamamlandı:**
- Toplam 999 örnek (jazz.00054.wav bozuk olduğu için atlandı — GTZAN'ın bilinen sorunu)
- Boyutlar: train (699, 128, 1292), val (150, 128, 1292), test (150, 128, 1292)
- Z-score normalize: mean ≈ 0, std ≈ 1, değer aralığı [-2.4, 2.8]
- Sınıflar her sette dengeli (stratified split)

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [5]:
class GTZANDataset(Dataset):
    """
    Partner'ın .npy dosyalarını PyTorch'a sarmalayan Dataset.
    augment=True iken SpecAugment uygular (sadece training için).
    """

    GENRES = ["blues", "classical", "country", "disco", "hiphop",
              "jazz", "metal", "pop", "reggae", "rock"]

    def __init__(self, X, y, augment=False):
        """
        X: (N, n_mels, T) — spectrogram array'i
        y: (N,) — etiket array'i (0-9)
        augment: SpecAugment uygulanacak mı?
        """
        self.X = X
        self.y = y
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # 1) Spectrogram'ı al — kopya alıyoruz çünkü augmentation onu değiştirecek
        spec = self.X[idx].copy().astype(np.float32)
        label = int(self.y[idx])

        # 2) SpecAugment (sadece training)
        if self.augment:
            # Time mask: zaman ekseninde rastgele dilim sıfırla
            if np.random.rand() < 0.5:  # %50 olasılıkla
                t_width = np.random.randint(5, 30)  # maske genişliği
                t_start = np.random.randint(0, max(1, spec.shape[1] - t_width))
                spec[:, t_start:t_start + t_width] = 0

            # Frequency mask: frekans ekseninde rastgele bant sıfırla
            if np.random.rand() < 0.5:
                f_width = np.random.randint(3, 15)
                f_start = np.random.randint(0, max(1, spec.shape[0] - f_width))
                spec[f_start:f_start + f_width, :] = 0

        # 3) NumPy → PyTorch tensor + kanal ekseni ekle
        # (128, 1292) → (1, 128, 1292)
        spec_tensor = torch.tensor(spec).unsqueeze(0)

        return spec_tensor, label

In [6]:
# Test: train için augmentation açık, val için kapalı
train_ds = GTZANDataset(X_train, y_train, augment=True)
val_ds   = GTZANDataset(X_val, y_val, augment=False)
test_ds  = GTZANDataset(X_test, y_test, augment=False)

print(f"Dataset uzunlukları:")
print(f"  Train: {len(train_ds)}")
print(f"  Val:   {len(val_ds)}")
print(f"  Test:  {len(test_ds)}")

# Tek bir örnek alalım
spec, label = train_ds[0]
print(f"\nİlk örnek:")
print(f"  Spec şekli: {spec.shape}")  # (1, 128, 1292)
print(f"  Spec tipi:  {spec.dtype}")  # torch.float32
print(f"  Etiket:     {label} ({GTZANDataset.GENRES[label]})")

# Aynı örneği iki kere alalım — augmentation farklı maskeler uygulamalı
spec1, _ = train_ds[0]
spec2, _ = train_ds[0]
print(f"\nİki ardışık çağrı aynı sonuç mu? {torch.equal(spec1, spec2)}")
print("(False olmalı — SpecAugment her seferde farklı maske)")

Dataset uzunlukları:
  Train: 699
  Val:   150
  Test:  150

İlk örnek:
  Spec şekli: torch.Size([1, 128, 1292])
  Spec tipi:  torch.float32
  Etiket:     9 (rock)

İki ardışık çağrı aynı sonuç mu? False
(False olmalı — SpecAugment her seferde farklı maske)


**Dataset sınıfı çalışıyor:**
- Üç set için Dataset oluşturuldu (train/val/test)
- Tek örnek formatı: `(1, 128, 1292)` float32 tensor + int etiket
- SpecAugment training setinde aktif (her çağrıda farklı maske doğrulandı)

In [7]:
BATCH_SIZE = 16

train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,           # her epoch'ta karıştır
    num_workers=2,          # paralel veri yükleme
    pin_memory=True,        # GPU transferi hızlı
    drop_last=True          # son eksik batch'i at (BatchNorm için temiz)
)

val_dl = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,          # val/test'te karıştırma
    num_workers=2,
    pin_memory=True
)

test_dl = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoader bilgisi:")
print(f"  Train: {len(train_dl)} batch × {BATCH_SIZE} = {len(train_dl)*BATCH_SIZE} örnek")
print(f"  Val:   {len(val_dl)} batch × {BATCH_SIZE} = ~{len(val_ds)} örnek")
print(f"  Test:  {len(test_dl)} batch × {BATCH_SIZE} = ~{len(test_ds)} örnek")

DataLoader bilgisi:
  Train: 43 batch × 16 = 688 örnek
  Val:   10 batch × 16 = ~150 örnek
  Test:  10 batch × 16 = ~150 örnek


In [8]:
# Tek bir batch al ve şeklini kontrol et
xb, yb = next(iter(train_dl))

print(f"Batch input shape:  {xb.shape}")   # [16, 1, 128, 1292]
print(f"Batch label shape:  {yb.shape}")   # [16]
print(f"Veri tipi (input):  {xb.dtype}")
print(f"Veri tipi (label):  {yb.dtype}")
print(f"Min/Max değer:      {xb.min():.3f} / {xb.max():.3f}")
print(f"Etiket örnekleri:   {yb.tolist()}")
print(f"Tür isimleri:       {[GTZANDataset.GENRES[i] for i in yb.tolist()]}")

Batch input shape:  torch.Size([16, 1, 128, 1292])
Batch label shape:  torch.Size([16])
Veri tipi (input):  torch.float32
Veri tipi (label):  torch.int64
Min/Max değer:      -2.394 / 2.754
Etiket örnekleri:   [2, 4, 5, 0, 8, 1, 6, 0, 3, 0, 7, 4, 5, 5, 3, 5]
Tür isimleri:       ['country', 'hiphop', 'jazz', 'blues', 'reggae', 'classical', 'metal', 'blues', 'disco', 'blues', 'pop', 'hiphop', 'jazz', 'jazz', 'disco', 'jazz']


**DataLoader hazır:**
- Batch boyutu: 16
- Train: 43 batch (drop_last=True ile 11 örnek atlandı, kalan 688 örnek)
- Val/Test: 10 batch × 16 = 160 (gerçek 150, küçük overflow yok)
- Output formatı: input `[16, 1, 128, 1292]` float32, label `[16]` int64
- Shuffle çalışıyor, etiketler dengeli karışık
